In [0]:
# Databricks notebook source
# Task 2: Excel (multi-sheet) -> Delta tables (one per sheet)
# - Lee el ultimo batch en Bronze usando Delivery-Notes-*.csv
# - Crea una Delta table por sheet y adiciona renglones

# COMMAND ----------
# Configuracion base (constantes)
STORAGE_ACCOUNT = "datablobstorage001"
CONTAINER = "blobstorage"
BRONZE_ROOT = "/bronze"
UC_CATALOG = "parts"
UC_SCHEMA = "bronze"

# COMMAND ----------
# Overrides para reproceso (opcionales)
dbutils.widgets.text("override_project", "")
dbutils.widgets.text("override_source", "")
dbutils.widgets.text("override_run_date", "")
dbutils.widgets.text("override_batch_id", "")
dbutils.widgets.text("override_bronze_base", "")
dbutils.widgets.text("override_bronze_excel_path", "")

# COMMAND ----------
# Helpers
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
from functools import reduce

abfss_base = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

def log(msg: str):
    ts = datetime.utcnow().isoformat()
    print(f"{ts} | {msg}")

def sanitize_identifier(name: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_]+", "_", name.strip())
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = "sheet"
    if s[0].isdigit():
        s = f"s_{s}"
    return s.lower()

def sanitize_columns(df):
    # Normaliza nombres de columnas para Delta (sin espacios ni caracteres inválidos)
    for col in df.columns:
        safe = sanitize_identifier(col)
        if safe != col:
            df = df.withColumnRenamed(col, safe)
    return df

def build_headers_from_first_row(df):
    # Toma la primera fila como header, sanitiza y deduplica
    first_row = df.limit(1).collect()
    if not first_row:
        return df, None
    raw_headers = list(first_row[0])
    headers = []
    seen = {}
    for h in raw_headers:
        name = "" if h is None else str(h).strip()
        name = sanitize_identifier(name) if name else "col"
        count = seen.get(name, 0) + 1
        seen[name] = count
        if count > 1:
            name = f"{name}_{count}"
        headers.append(name)
    return df, headers

def upsert_by_part_number(df, table_name: str, delta_path: str):
    if "part_number" not in df.columns:
        raise ValueError("La columna 'part_number' no existe en el sheet. No se puede hacer upsert.")

    df = df.filter(F.col("part_number").isNotNull()).dropDuplicates(["part_number"])

    # Habilita auto-merge de schema para nuevas columnas
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

    if spark.catalog.tableExists(table_name):
        log(f"Upsert en tabla existente: {table_name}")
        delta_tbl = DeltaTable.forName(spark, table_name)
        (delta_tbl.alias("t")
            .merge(df.alias("s"), "t.part_number = s.part_number")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        log(f"Creando tabla nueva: {table_name} @ {delta_path}")
        (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .save(delta_path))
        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            USING DELTA
            LOCATION '{delta_path}'
            """
        )
        # Habilitar CDF
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

def get_task_value(task_key: str, key: str):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key)
    except Exception:
        return None

# COMMAND ----------
# Resolver batch via Task Values (si existe). Sin fallback.

# Debe coincidir con el task_key real en el Job
task_key = "autoloader_landing_to_bronze_workflow"
bronze_base = get_task_value(task_key, "bronze_base")
bronze_excel_path = get_task_value(task_key, "bronze_excel_path")
batch_id = get_task_value(task_key, "batch_id")
run_date = get_task_value(task_key, "run_date")
project = get_task_value(task_key, "project")
source = get_task_value(task_key, "source")

# Overrides para reproceso (si se pasan en el Job)
override_bronze_base = dbutils.widgets.get("override_bronze_base").strip()
override_bronze_excel_path = dbutils.widgets.get("override_bronze_excel_path").strip()
override_project = dbutils.widgets.get("override_project").strip()
override_source = dbutils.widgets.get("override_source").strip()
override_run_date = dbutils.widgets.get("override_run_date").strip()
override_batch_id = dbutils.widgets.get("override_batch_id").strip()

if override_bronze_base or override_bronze_excel_path:
    if not override_bronze_base or not override_bronze_excel_path:
        raise ValueError("Para override se requieren override_bronze_base y override_bronze_excel_path.")
    bronze_base = override_bronze_base
    bronze_excel_path = override_bronze_excel_path
    project = override_project or project
    source = override_source or source
    run_date = override_run_date or run_date
    batch_id = override_batch_id or batch_id
    log("Using override parameters for reproceso.")
elif bronze_base and bronze_excel_path:
    log(f"Using Task Values from '{task_key}'.")
else:
    raise ValueError(
        "No Task Values disponibles del task autoloader_landing_to_bronze y no se proporcionaron overrides."
    )

log(f"Resolved Project={project}, Source={source}, Date={run_date}, Batch={batch_id}")
log(f"Bronze Excel path: {bronze_excel_path}")

# COMMAND ----------
# Procesar cada Excel en Bronze y crear tablas por sheet

def ensure_path_exists(path: str, label: str):
    try:
        dbutils.fs.ls(path)
    except Exception as e:
        raise FileNotFoundError(f"No se pudo acceder a {label}: {path}") from e

ensure_path_exists(bronze_base, "bronze_base")
ensure_path_exists(bronze_excel_path, "bronze_excel_path")

excel_listing = dbutils.fs.ls(bronze_excel_path)
excel_files = [f.path for f in excel_listing if f.path.lower().endswith(".xlsx")]

if not excel_files:
    sample = [f.path.split("/")[-1] for f in excel_listing[:10]]
    raise FileNotFoundError(
        f"No Excel (.xlsx) files found in Bronze: {bronze_excel_path}. "
        f"Files present (sample): {sample}"
    )

# Validar que los archivos Excel no esten vacios
empty_excels = [f.path for f in excel_listing if f.path.lower().endswith(".xlsx") and f.size == 0]
if empty_excels:
    raise ValueError(f"Se encontraron Excel vacios: {empty_excels}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")

try:
    from openpyxl import load_workbook
except Exception as e:
    raise ImportError(
        "No se pudo importar openpyxl. Instala la libreria en el cluster para leer sheets."
    ) from e

for excel_file in excel_files:
    file_name = excel_file.split("/")[-1]
    file_stem = file_name.rsplit(".", 1)[0]
    safe_file = sanitize_identifier(file_stem)
    # bronze_base ya incluye abfss_base cuando viene de Task Values o fallback
    # Escribe Delta en una ruta dedicada por sheet (nombre estandar)
    delta_base = f"{abfss_base}{BRONZE_ROOT}/_delta/by_sheet"

    local_xlsx = f"/tmp/excel_{uuid.uuid4().hex}.xlsx"
    dbutils.fs.cp(excel_file, f"file:{local_xlsx}")

    wb = load_workbook(local_xlsx, read_only=True, data_only=True)
    sheet_names = wb.sheetnames
    wb.close()

    if not sheet_names:
        log(f"Excel sin sheets: {excel_file}")
        continue

    log(f"Sheets detectadas en {file_name}: {sheet_names}")

    for sheet in sheet_names:
        safe_sheet = sanitize_identifier(sheet)
        table_name = f"{UC_CATALOG}.{UC_SCHEMA}.{safe_sheet}"
        delta_path = f"{delta_base}/{safe_sheet}"

        log(f"Procesando sheet '{sheet}' -> {table_name} @ {delta_path}")

        df = (
            spark.read
            .format("com.crealytics.spark.excel")
            .option("dataAddress", f"'{sheet}'!A1")
            .option("header", "false")
            .option("inferSchema", "false")
            .option("treatEmptyValuesAsNulls", "true")
            .load(excel_file)
        )

        df, headers = build_headers_from_first_row(df)
        if headers is None:
            log(f"Sheet vacia (sin filas): {excel_file} :: {sheet}")
            continue
        log(f"Headers detectados ({len(headers)}): {headers}")
        # Remueve la fila de headers del dataset
        schema = StructType([StructField(h, StringType(), True) for h in headers])
        df = (
            df.rdd
            .zipWithIndex()
            .filter(lambda r: r[1] > 0)
            .map(lambda r: r[0])
            .toDF(schema)
        )
        log(f"Filas despues de remover header: {df.count()}")

        # Concatenar todas las columnas en un solo texto para embeddings
        cols_as_str = [F.col(c).cast("string") for c in df.columns]
        df = df.withColumn("__text", F.concat_ws(" | ", *cols_as_str))

        if df.rdd.isEmpty():
            log(f"Sheet vacia (sin filas): {excel_file} :: {sheet}")
            continue

        df = (
            df
            .withColumn("__sheet_name", F.lit(sheet))
            .withColumn("__source_file", F.lit(excel_file))
            .withColumn("__project", F.lit(project))
            .withColumn("__source", F.lit(source))
            .withColumn("__run_date", F.lit(run_date))
            .withColumn("__batch_id", F.lit(batch_id))
            .withColumn("__ingest_ts", F.current_timestamp())
        )

        upsert_by_part_number(df, table_name, delta_path)

log("Task 2 completed.")
